In [1]:
# Ermöglicht das erneute Laden der .py Module, auch nach deren Modifikation ohne den Kernel neu starten zu müssen

%load_ext autoreload
%autoreload 2


In [8]:
import json
import sqlite3
import requests
import re
from html import unescape
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
from urllib.parse import urlparse
from src.tagesschau_client import TagesschauClient
import os
import hashlib
import uuid



In [40]:
#from tagesschau_client import TagesschauClient

from dotenv import load_dotenv
load_dotenv()

from src.tagesschau_client import TagesschauClient

client = TagesschauClient(
    api_config_path="config/api_config.json",
    regions_path="config/regions.json",
    source_regions_path="config/source_regions.json",
    url_region_keywords_path="config/url_region_keywords.json",
    filters_path="config/filters.json",
)

await client.collect_and_store()



🕒 Ingest watermark (from ingest_date): 2026-01-17T18:21:35

📊 TAGESSCHAU INGEST SUMMARY
🔹 Artikel von API (Index): 102
🕒 Nach Watermark relevant: 74
💾 Artikel gespeichert:     74
📄 Kein Fulltext:           0
❌ Fehlgeschlagen:          0
⏭️ Gefiltert (Typ):         28
⏭️ Gefiltert (Ressort):     0
⏭️ Gefiltert (Watermark):   0


In [43]:
import os
import asyncio
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from pathlib import Path
import libsql_client

# ----------------------------
# Config
# ----------------------------
QDRANT_PATH = Path("vector_store/qdrant_turso_test4")
COLLECTION = "turso_articles"
LIMIT = 5

# ----------------------------
# Env
# ----------------------------
TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

# ----------------------------
# Embedding model
# ----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = 384

# ----------------------------
# Qdrant
# ----------------------------
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
qdrant = QdrantClient(path=str(QDRANT_PATH))

if COLLECTION not in [c.name for c in qdrant.get_collections().collections]:
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )

# ----------------------------
# Load 5 articles from Turso
# ----------------------------
async def main():
    print("🔌 Connecting to Turso...")
    db = libsql_client.create_client(
        url=TURSO_DB_URL,
        auth_token=TURSO_AUTH_TOKEN,
    )

    print("📥 Loading 5 articles from DB...")

    rs = await db.execute("""
    SELECT
        external_id,
        title,
        fulltext
    FROM articles
    WHERE
        title LIKE '%Auto%'
        OR title LIKE '%BMW%'
        OR title LIKE '%VW%'
        OR title LIKE '%Tesla%'
        OR title LIKE '%Wirtschaft%'
        OR title LIKE '%Industrie%'
    LIMIT 10
""")


    rows = rs.rows
    print(f"Loaded {len(rows)} articles.")

    # ----------------------------
    # Build embeddings
    # ----------------------------
    texts = []
    metadatas = []

    for r in rows:
        text = (r["title"] or "") + "\n\n" + (r["fulltext"] or "")
        texts.append(text)
        metadatas.append({
            "external_id": r["external_id"],
            "title": r["title"],
        })

    print("🧠 Computing embeddings...")
    vectors = model.encode(texts)

    # ----------------------------
    # Insert into Qdrant
    # ----------------------------
    points = []
    for i, (vec, meta) in enumerate(zip(vectors, metadatas)):
        points.append(
            PointStruct(
                id=i,
                vector=vec.tolist(),
                payload=meta,
            )
        )

    qdrant.upsert(collection_name=COLLECTION, points=points)

    print("✅ Inserted into Qdrant.")

    # ----------------------------
    # Query
    # ----------------------------
    query_text = "Was gibt es Neues zu Autoherstellern und Wirtschaft?"

    print("\n🔍 Query:", query_text)

    qvec = model.encode([query_text])[0]

    res = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec.tolist(),
        limit=5,
    )

    print("\n📄 Results:\n")

    for r in res.points:
        print(f"Score: {r.score:.4f}")
        print("external_id:", r.payload["external_id"])
        print("title:", r.payload["title"])
        print("-" * 60)

    await db.close()


if __name__ == "__main__":
    await main()


🔌 Connecting to Turso...
📥 Loading 5 articles from DB...
Loaded 10 articles.
🧠 Computing embeddings...
✅ Inserted into Qdrant.

🔍 Query: Was gibt es Neues zu Autoherstellern und Wirtschaft?

📄 Results:

Score: 0.3183
external_id: tagesschau_fm-story-swr-2c3825fd-4d8c-3e60-b189-39175f3808ca
title: Auto steht quer: Sperrung auf A81 nach Unfall bei Neuenstadt
------------------------------------------------------------
Score: 0.3122
external_id: c4f1fdef-8f2c-4cfe-86aa-7b2e1657f0e0
title: IHK zu Schwerin zieht negative Wirtschaftsbilanz für 2025
------------------------------------------------------------
Score: 0.3020
external_id: af069799-d332-434c-b892-220241325026
title: Landwirte wollen in MV erneut Autobahnauffahrten blockieren
------------------------------------------------------------
Score: 0.2985
external_id: tagesschau_fm-story-rbb_brandenburg-landwirte-proteste-autobahn-auffahrten-mercosur
title: Landwirte in Brandenburg kündigen Proteste an Autobahnen an
--------------------

In [50]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()

qdrant = QdrantClient(
    url=os.environ["QADRANT_ENDPOINT"],
    api_key=os.environ["QADRANT_API_KEY"]
)



In [53]:
print(qdrant.get_collections())


collections=[]


In [54]:
from sentence_transformers import SentenceTransformer

class LocalEmbedder:
    def __init__(self, model_name="BAAI/bge-m3"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        texts = [f"passage: {t}" for t in texts]
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode([f"query: {text}"])[0].tolist()



In [44]:
import os
import uuid
import json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import libsql_client
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from openai import OpenAI

# ----------------------------
# Env
# ----------------------------
load_dotenv()

TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

QDRANT_URL = os.environ["QADRANT_ENDPOINT"]
QDRANT_API_KEY = os.environ["QADRANT_API_KEY"]

OPENAI_API_KEY = os.environ["OPEN_AI_KEY"]

# ----------------------------
# Paths
# ----------------------------
SCHEMA_PATH = "config/article_schema.json"
PROMPTS_PATH = "config/prompts.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------
# Index / Kosten-Schutz Config
# ----------------------------
COLLECTION = "dummy_II"
EMBED_DIM = 1536

BATCH_SIZE = 100
MAX_ARTICLES = 2000
DRY_RUN = False

FILTER_KEYWORDS = ["Iran"]

MIN_DATE = "2026-01-14"
MAX_DATE = "2026-01-15"

# ----------------------------
# Helpers
# ----------------------------
def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_result_to_json(data: dict, prefix: str = "rag_result") -> Path:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = OUTPUT_DIR / f"{prefix}_{ts}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"💾 Saved result to: {path}")
    return path

# ----------------------------
# OpenAI Clients
# ----------------------------
class OpenAIEmbedder:
    def __init__(self, api_key: str, model="text-embedding-3-small"):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        resp = self.client.embeddings.create(model=self.model, input=texts)
        return [d.embedding for d in resp.data]

    def embed_query(self, text: str) -> list[float]:
        resp = self.client.embeddings.create(model=self.model, input=[text])
        return resp.data[0].embedding


class OpenAISummarizer:
    def __init__(self, api_key: str, prompts: dict, prompt_key: str, model="gpt-4.1-mini"):
        self.client = OpenAI(api_key=api_key)
        self.model = model

        if prompt_key not in prompts:
            raise ValueError(f"Prompt key '{prompt_key}' not found in prompts JSON")

        self.system_template = prompts[prompt_key]["system"]
        self.user_template = prompts[prompt_key]["user"]

    def summarize(self, query: str, documents: list[dict]) -> str:
        blocks = []
        for i, doc in enumerate(documents, 1):
            blocks.append(
                f"[{i}]\n"
                f"Titel: {doc.get('title','')}\n"
                f"Datum: {doc.get('published_at','')}\n"
                f"URL: {doc.get('url','')}\n"
                f"Inhalt:\n{doc.get('text','')}\n"
            )

        documents_text = "\n\n".join(blocks)

        system_prompt = self.system_template
        user_prompt = self.user_template.format(query=query, documents=documents_text)

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.2,
        )

        out = resp.choices[0].message.content or ""
        return out.strip()

# ----------------------------
# Schema Handling
# ----------------------------
def get_columns_by_role(schema: dict, role: str) -> list[str]:
    return [col for col, spec in schema["columns"].items() if spec["role"] == role]

def get_select_columns(schema: dict) -> list[str]:
    return [col for col, spec in schema["columns"].items() if spec["role"] != "ignore"]

def build_select_sql(schema: dict, where_clause: str, limit: int, offset: int) -> str:
    cols = get_select_columns(schema)
    table = schema["table"]
    return f"""
        SELECT {", ".join(cols)}
        FROM {table}
        WHERE {where_clause}
        LIMIT {limit} OFFSET {offset}
    """

def build_embedding_text(row, schema: dict) -> str:
    parts = []
    for col in get_columns_by_role(schema, "embedding"):
        val = row[col]
        if val:
            parts.append(str(val))
    return "\n\n".join(parts)

def build_payload(row, schema: dict) -> dict:
    payload = {}
    for col in get_columns_by_role(schema, "payload"):
        payload[col] = row[col]
    return payload

# ----------------------------
# SQL Filter
# ----------------------------
def build_where_clause():
    clauses = ["fulltext IS NOT NULL"]

    if MIN_DATE:
        clauses.append(f"published_at >= '{MIN_DATE}'")
    if MAX_DATE:
        clauses.append(f"published_at <= '{MAX_DATE}'")

    if FILTER_KEYWORDS:
        like_parts = []
        for kw in FILTER_KEYWORDS:
            kw = kw.replace("'", "''")
            like_parts.append(f"title LIKE '%{kw}%'")
            like_parts.append(f"fulltext LIKE '%{kw}%'")
        clauses.append("(" + " OR ".join(like_parts) + ")")

    return " AND ".join(clauses)

# ----------------------------
# Load full texts for RAG
# ----------------------------
async def fetch_articles_by_ids(db, ids: list[str]) -> list[dict]:
    if not ids:
        return []

    ids_sql = ", ".join([f"'{i}'" for i in ids])

    sql = f"""
        SELECT external_id, title, published_at, url, fulltext
        FROM articles
        WHERE external_id IN ({ids_sql})
    """

    rs = await db.execute(sql)

    docs = []
    for r in rs.rows:
        docs.append({
            "external_id": r["external_id"],
            "title": r["title"],
            "published_at": r["published_at"],
            "url": r["url"],
            "text": r["fulltext"],
        })

    return docs

# ----------------------------
# Main
# ----------------------------
async def main():
    schema = load_json(SCHEMA_PATH)
    prompts = load_json(PROMPTS_PATH)

    embedder = OpenAIEmbedder(OPENAI_API_KEY)
    summarizer = OpenAISummarizer(
        api_key=OPENAI_API_KEY,
        prompts=prompts,
        prompt_key="news_summary",
    )

    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    if not qdrant.collection_exists(COLLECTION):
        qdrant.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )

    print("🔌 Connecting to Turso...")
    db = libsql_client.create_client(url=TURSO_DB_URL, auth_token=TURSO_AUTH_TOKEN)

    where_clause = build_where_clause()
    print("🔍 WHERE:", where_clause)

    offset = 0
    total_indexed = 0

    # ----------------------------
    # Index
    # ----------------------------
    while True:
        if total_indexed >= MAX_ARTICLES:
            break

        sql = build_select_sql(schema, where_clause, BATCH_SIZE, offset)
        rs = await db.execute(sql)
        rows = rs.rows

        if not rows:
            break

        texts, payloads, ids = [], [], []

        for r in rows:
            if total_indexed >= MAX_ARTICLES:
                break

            text = build_embedding_text(r, schema)
            if not text.strip():
                continue

            payload = build_payload(r, schema)

            texts.append(text)
            payloads.append(payload)

            pid = str(uuid.uuid5(uuid.NAMESPACE_URL, str(payload["external_id"])))
            ids.append(pid)

            total_indexed += 1

        if not texts:
            offset += BATCH_SIZE
            continue

        if not DRY_RUN:
            vectors = embedder.embed_documents(texts)
            points = [
                PointStruct(id=ids[i], vector=vectors[i], payload=payloads[i])
                for i in range(len(ids))
            ]
            qdrant.upsert(collection_name=COLLECTION, points=points)

        offset += BATCH_SIZE

    # ----------------------------
    # Retrieval
    # ----------------------------
    keywords_str = " ".join(FILTER_KEYWORDS)
    query_text = f"Neuigkeiten zu {keywords_str}"

    qvec = embedder.embed_query(query_text)

    res = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec,
        limit=5,
    )

    retrieved = res.points
    external_ids = [p.payload["external_id"] for p in retrieved]

    docs = await fetch_articles_by_ids(db, external_ids)

    # ----------------------------
    # Summary (MUSS vor JSON Build passieren)
    # ----------------------------
    summary = summarizer.summarize(query_text, docs)
    print(f"🧾 Summary length: {len(summary)} chars")

    # ----------------------------
    # JSON Result (enthält Summary garantiert)
    # ----------------------------
    result = {
        "query": query_text,
        "filters": {
            "keywords": FILTER_KEYWORDS,
            "min_date": MIN_DATE,
            "max_date": MAX_DATE,
        },
        "retrieved_documents": [
            {
                "rank": i,
                "score": float(p.score),
                "external_id": p.payload.get("external_id"),
                "title": p.payload.get("title"),
                "published_at": p.payload.get("published_at"),
                "url": p.payload.get("url"),
            }
            for i, p in enumerate(retrieved, 1)
        ],
        "sources": [
            {
                "index": i,
                "external_id": d["external_id"],
                "title": d["title"],
                "published_at": d["published_at"],
                "url": d["url"],
            }
            for i, d in enumerate(docs, 1)
        ],
        "summary": summary,
        "summary_text": summary,  # doppelt für Debug/Frontend-Komfort
        "meta": {
            "collection": COLLECTION,
            "model_embedding": "text-embedding-3-small",
            "model_summary": "gpt-4.1-mini",
            "created_at": datetime.now().isoformat(),
        },
    }

    out_path = save_result_to_json(result, prefix="news_rag")

    print("\n🧠 ZUSAMMENFASSUNG:\n")
    print(summary)

    print("\n📌 JSON CHECK:")
    print("file:", out_path)
    print("summary_in_json:", bool(result["summary"]))

    await db.close()

# ----------------------------
# Notebook Entrypoint
# ----------------------------
await main()


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff65f3d250>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff64539310>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff643ba950>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff6460b4d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff5ffc18d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff64267990>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff64788490>


🔌 Connecting to Turso...
🔍 WHERE: fulltext IS NOT NULL AND published_at >= '2026-01-14' AND published_at <= '2026-01-15' AND (title LIKE '%Iran%' OR fulltext LIKE '%Iran%')
🧾 Summary length: 1634 chars
💾 Saved result to: output/news_rag_20260120_213139.json

🧠 ZUSAMMENFASSUNG:

Im Iran dauern die landesweiten Proteste gegen das Regime an, trotz massiver Gewalt durch Sicherheitskräfte, die nach Angaben von Menschenrechtsorganisationen bereits Tausende Demonstranten getötet haben; verifizierte Zahlen liegen bei mindestens 734 bis über 3.400 Toten, wobei die tatsächliche Zahl vermutlich höher ist [1][3][5]. Das Land ist seit Tagen vom Internet weitgehend abgeschnitten, was die Kommunikation und unabhängige Berichterstattung erschwert; einige Menschen nutzen Satelliteninternetdienste wie Starlink, deren Nutzung jedoch verboten und von den Behörden bekämpft wird [1][3][5]. Die iranische Justiz hat Schnellverfahren gegen Festgenommene eingeleitet und droht mit Hinrichtungen, insbesondere unt